In [1]:
# Block 0 — setup. Code from GitHub onto the VM disk; data + embeddings on Drive.
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

# numpy/scipy/torch are filtered out: Colab's torch is built against its own numpy
!grep -viE '^(torch|torchvision|torchaudio|nvidia-|numpy|scipy|#)' requirements.txt > req_colab.txt
!pip install -q -r req_colab.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 12 (delta 8), reused 12 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 634.37 KiB | 764.00 KiB/s, done.
From https://github.com/Tienanh209/NeoAntigen
   0b59d6d..3634baf  main       -> origin/main
Updating 0b59d6d..3634baf
Fast-forward
 CLAUDE.md                 |    1 +
 embeddings_authors.ipynb  |  138 ++
 experiments.ipynb         | 5571 ++++++++++++++++++++++++++++++++++++++++++++-
 experiments_authors.ipynb |  314 +++
 src/authors.py            |   91 +
 src/data.py               |    8 +
 src/results.py            |   53 +-
 src/runner.py             |    2 +
 src/train.py              |    5 +-
 9 files changed, 6085 insertions(+), 98 deletions(-)
 create mode 100644 embeddings_authors.ipy

In [2]:
# Block 1 — the AUTHORS' dataset, embedding caches, shared settings.
# Every experiment block below depends on THIS cell and on nothing else:
# they communicate only through the results store on Drive.
import pandas as pd
from src import authors, config, data, embed, results, runner

df = authors.load()      # github.com/B1607/DeepNeoAG at a pinned commit, cached on Drive
BID = data.build_id(df)
MAXLEN = data.max_len(df)
SEEDS = (42,)               # single seed: fast, but AUC_std is then NaN and nothing
                            # separates a real effect from run-to-run noise
PLM = "prott5"              # feature set for the hyperparameter sweeps, as in the paper
ESM = "esm2_650m"           # the ESM model used by the ESM baselines
FORCE = False               # True re-runs committed configurations

for plm in ("prott5", "esm2_650m", "esm1b"):
    if embed.is_cached(plm, BID):
        embed.stage(plm, BID)
    else:
        print(f"!! {plm} not cached — run embeddings.ipynb first")

print("build id:", BID, "| max_len:", MAXLEN, "| results:", config.RESULTS_DIR)
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

authors dataset: 7012 peptides -> /content/drive/MyDrive/s1156052/NeoAntigen/data/authors_dataset.csv
!! prott5 not cached — run embeddings.ipynb first
!! esm2_650m not cached — run embeddings.ipynb first
!! esm1b not cached — run embeddings.ipynb first
build id: bdfb2e744c | max_len: 35 | results: /content/drive/MyDrive/s1156052/NeoAntigen/ckpt/results
label     0    1
split           
test   1343   61
train  5367  241


In [4]:
# Block 2 — Experiment A: single window sizes (paper Table 2)
cfgs = [dict(label=f"window {w}", features={"kind": "plm", "plm": PLM},
             model={"kind": "mcnn", "windows": [w], "filters": 512})
        for w in (8, 16, 24, 32, 34)]
runner.run_series("AU_A_window", cfgs, df=df, seeds=SEEDS, force=FORCE)

print("\nIndependent test")
display(results.fmt(results.table("AU_A_window")))
print("5-fold cross-validation")
display(results.fmt(results.table("AU_A_window", eval_split="cv")))
results.roc("AU_A_window", title="Single window sizes — independent test");

[AU_A_window 1/5] window 8


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/s1156052/NeoAntigen/data/emb/bdfb2e744c/prott5/vectors.npy'

In [ ]:
# Block 3 — Experiment B: window combinations (paper Table 3)
combos = [[34], [34, 32], [34, 32, 24], [34, 32, 24, 16], [34, 32, 24, 16, 8]]
cfgs = [dict(label=str(c), features={"kind": "plm", "plm": PLM},
             model={"kind": "mcnn", "windows": c, "filters": 512}) for c in combos]
runner.run_series("AU_B_combo", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("AU_B_combo")))
display(results.fmt(results.table("AU_B_combo", eval_split="cv")))
results.roc("AU_B_combo", title="Window combinations — independent test");

In [ ]:
# Block 4 — Experiment C: filters per window (paper Table 4)
# Only the filter count changes: dropout, hidden width and batch size stay at their
# defaults, so differences are due to filters alone. conv_params is in the table on
# purpose: at 2048 filters the convolutions hold ~200-300M parameters for ~1.1k
# training peptides, which is the point of this sweep.
WINDOWS = [34, 32, 24, 16]
cfgs = [dict(label=f"{f} filters", features={"kind": "plm", "plm": PLM},
             model={"kind": "mcnn", "windows": WINDOWS, "filters": f})
        for f in (128, 256, 512, 1024, 2048)]
runner.run_series("AU_C_filters", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("AU_C_filters", extra_cols=("conv_params",))))
results.roc("AU_C_filters", title="Filters per window — independent test");

In [ ]:
# Block 5 — Experiment J: hidden nodes (width of the fully connected layer)
# Only the hidden width changes; windows, filters and dropout stay at the defaults.
WINDOWS = [34, 32, 24, 16]
cfgs = [dict(label=f"{h} hidden", features={"kind": "plm", "plm": PLM},
             model={"kind": "mcnn", "windows": WINDOWS, "filters": 512, "hidden": h})
        for h in (32, 64, 128, 256, 512, 1024)]
runner.run_series("AU_J_hidden", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("AU_J_hidden")))
display(results.fmt(results.table("AU_J_hidden", eval_split="cv")))
results.roc("AU_J_hidden", title="Hidden nodes — independent test");

In [ ]:
# Block 6 — Experiment K: dropout (applied before and after the hidden layer)
# Only the dropout rate changes. Watch cv vs test: with ~100 training positives, too
# little dropout shows up as a good cv score that does not carry over to test.
WINDOWS = [34, 32, 24, 16]
cfgs = [dict(label=f"dropout {p}", features={"kind": "plm", "plm": PLM},
             model={"kind": "mcnn", "windows": WINDOWS, "filters": 512, "dropout": p})
        for p in (0.0, 0.1, 0.3, 0.5, 0.7)]
runner.run_series("AU_K_dropout", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("AU_K_dropout")))
display(results.fmt(results.table("AU_K_dropout", eval_split="cv")))
results.roc("AU_K_dropout", title="Dropout — independent test");

In [ ]:
# Block 7 — Experiment D: pre-trained language model features (paper Table 5, Fig 4a)
BEST_WINDOWS, BEST_FILTERS = [34, 32, 24, 16], 512
cfgs = [dict(label=name, features={"kind": "plm", "plm": plm},
             model={"kind": "mcnn", "windows": BEST_WINDOWS, "filters": BEST_FILTERS})
        for plm, name in [("prott5", "ProtTrans"), ("esm2_650m", "ESM-2"), ("esm1b", "ESM-1b")]]
runner.run_series("AU_D_plm", cfgs, df=df, seeds=SEEDS, force=FORCE)

print("Cross-validation (5 folds)")
display(results.fmt(results.table("AU_D_plm", eval_split="cv")))
print("Independent test")
display(results.fmt(results.table("AU_D_plm")))
results.roc("AU_D_plm", title="PLM features — independent test");

In [ ]:
# Block 8 — Experiment E: classifiers (paper Table 6, Fig 4b)
# KNN / RF / SVM take the mean-pooled embedding; the mCNN keeps per-residue input.
cfgs = [dict(label=n.upper(), features={"kind": "plm_pooled", "plm": PLM},
             model={"kind": "sklearn", "name": n}) for n in ("knn", "rf", "svm")]
cfgs.append(dict(label="mCNN", features={"kind": "plm", "plm": PLM},
                 model={"kind": "mcnn", "windows": [34, 32, 24, 16], "filters": 512}))
runner.run_series("AU_E_clf", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("AU_E_clf")))
results.roc("AU_E_clf", title="Classifiers — independent test");

In [ ]:
# Block 9 — Experiment F: length confound (replaces the paper's MHC-I / MHC-II table)
# This dataset carries no MHC annotation, and a length rule puts only a handful of the
# 61 test positives in the 8-11 aa range, so that table cannot be rebuilt here. More
# important is what the length distribution shows: no peptide is shorter than 11 aa
# (consistent with CD-HIT's default -l 10, which drops them) and positives are much
# longer than negatives. If length alone separates the classes, a good AUC from a
# sequence model proves much less than it appears to.
print(df.groupby(["split", "label"])["length"].describe()[["count", "mean", "min", "50%", "max"]].round(1))

cfgs = [dict(label="length only (LogReg)", features={"kind": "length"},
             model={"kind": "sklearn", "name": "logreg"})]
runner.run_series("AU_F_length", cfgs, df=df, seeds=SEEDS, force=FORCE)
display(results.fmt(results.table("AU_F_length")))
display(results.fmt(results.table("AU_F_length", eval_split="cv")))

In [ ]:
# Block 10 — Experiment G: baselines
# Composition alone -> order (one-hot mCNN) -> pretraining (ESM + LogReg) -> local
# scanning on top of pretraining (ESM + CNN, ESM + mCNN). All three ESM rows share the
# same ESM-2 embeddings, so the gaps isolate what pooling, one window and several
# windows add. "length only" is a control: if peptide length alone separates the
# classes, a good AUC from a sequence model proves much less than it appears to.
WINDOWS = [34, 32, 24, 16]
cfgs = [
    dict(label="length only (LogReg)", features={"kind": "length"},
         model={"kind": "sklearn", "name": "logreg"}),
    dict(label="Amino acid + LogReg", features={"kind": "aac"},
         model={"kind": "sklearn", "name": "logreg"}),
    dict(label="One-hot + mCNN", features={"kind": "onehot"},
         model={"kind": "mcnn", "windows": WINDOWS, "filters": 512}),
    dict(label="ESM + LogReg", features={"kind": "plm_pooled", "plm": ESM},
         model={"kind": "sklearn", "name": "logreg"}),
    dict(label="ESM + CNN", features={"kind": "plm", "plm": ESM},
         model={"kind": "mcnn", "windows": [34], "filters": 512}),
    dict(label="ESM + mCNN", features={"kind": "plm", "plm": ESM},
         model={"kind": "mcnn", "windows": WINDOWS, "filters": 512}),
]
runner.run_series("AU_G_baselines", cfgs, df=df, seeds=SEEDS, force=FORCE)

display(results.fmt(results.table("AU_G_baselines")))
results.roc("AU_G_baselines", title="Baselines — independent test");

In [ ]:
# Block 11 — Experiment H: class weighting (CS535 E7)
# At ~1 positive per 13 negatives, this is what moves sensitivity against precision.
base = dict(features={"kind": "plm", "plm": PLM},
            model={"kind": "mcnn", "windows": [34, 32, 24, 16], "filters": 512})
cfgs = [dict(label="weighted BCE", train_cfg={"class_weight": True}, **base),
        dict(label="unweighted BCE", train_cfg={"class_weight": False}, **base)]
runner.run_series("AU_H_weight", cfgs, df=df, seeds=SEEDS, force=FORCE)

print("Tuned threshold (chosen on CV)")
display(results.fmt(results.table("AU_H_weight")))
print("Fixed 0.5 threshold — note how differently 0.5 behaves between the two")
display(results.fmt(results.table("AU_H_weight", thr_mode="fixed")))

In [ ]:
# Block 12 — Experiment I: similarity leakage (CS535 E8)
# Random folds let near-duplicate peptides sit on both sides of the split;
# grouped folds keep peptides that share an 8-mer together.
base = dict(features={"kind": "plm", "plm": PLM},
            model={"kind": "mcnn", "windows": [34, 32, 24, 16], "filters": 512})
cfgs = [dict(label="random folds", split_mode="stratified", **base),
        dict(label="similarity-controlled folds", split_mode="grouped", **base)]
runner.run_series("AU_I_leak", cfgs, df=df, seeds=SEEDS, force=FORCE)

print("Cross-validation — the leakage shows up here, not on the test split")
display(results.fmt(results.table("AU_I_leak", eval_split="cv")))
display(results.fmt(results.table("AU_I_leak")))

g = data.kmer_groups(df)
print(f"\n{len(df)} peptides fall into {g.nunique()} shared-8-mer groups; "
      f"largest group has {g.value_counts().iloc[0]}")

In [ ]:
# Block 13 — report: every table and figure, rebuilt from the store. No training.
# This is what to run after a disconnect to see the whole state of the project.
TITLES = {"AU_G_baselines": "Baselines", "AU_F_length": "Length-only control", "AU_E_clf": "Classifiers", "AU_D_plm": "PLM features",
          "AU_A_window": "Single window sizes", "AU_B_combo": "Window combinations",
          "AU_C_filters": "Filters per window", "AU_J_hidden": "Hidden nodes", "AU_K_dropout": "Dropout",
          "AU_H_weight": "Class weighting", "AU_I_leak": "Similarity leakage"}

for exp_id, title in TITLES.items():
    t = results.table(exp_id)
    if t.empty:
        print(f"--- {title}: not run yet\n")
        continue
    print(f"--- {title} ({exp_id}) — independent test")
    display(results.fmt(t))
    results.roc(exp_id, title=f"{title} — independent test")

runs = results.load_runs()
print(f"\n{len(runs)} committed runs in {config.RESULTS_DIR}")